In [ ]:
!pip install transformers datasets bayesian-optimization optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.4/383.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.5 MB/s eta 0:00:00


## Using Pipeline

In [ ]:
# Import libraries
from transformers import pipeline, set_seed
import optuna

# Set up the text generation pipeline (e.g., GPT-2 model)
generator = pipeline("text-generation", model="gpt2")
set_seed(42)  # Ensure reproducibility

# Define the objective function for Bayesian Optimization
def objective(trial):
    # Sample hyperparameters for optimization
    max_length = trial.suggest_int("max_length", 20, 100)  # Length of the output
    temperature = trial.suggest_float("temperature", 0.5, 1.5)  # Creativity level

    # Define a fixed prompt for text generation
    prompt = "Once upon a time in a world of artificial intelligence,"

    # Generate text
    generated_text = generator(
        prompt,
        max_length=max_length,
        temperature=temperature,
        num_return_sequences=1,
    )[0]["generated_text"]

    # Evaluate the result (toy example: we measure "length of generated text")
    # In real scenarios, this could be evaluated using BLEU, ROUGE, or other metrics
    length_score = len(generated_text)  # Use length as a simple proxy for "quality"
    return -abs(length_score - 80)  # Targeting length around 80 characters (negated for minimization)

# Optimize using Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

# Display the best hyperparameters and score
print("Best score:", study.best_value)
print("Best hyperparameters:", study.best_params)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu
[I 2025-01-26 15:41:27,257] A new study created in memory with name: no-name-cd2e336b-9e39-431b-b53b-2a8e16b5b7c0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[I 2025-01-26 15:41:30,482] Trial 0 finished with value: -110.0 and parameters: {'max_length': 41, 'temperature': 0.5122921589669048}. Best is trial 0 with value: -110.0.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[I 2025-01-26 15:41:40,282] Trial 1 finished with value: -322.0 and parameters: {'max_length': 79, 'temperature': 1.3195804165047593}. Best is trial 0 with value: -110.0.
Setti

Best score: -10.0
Best hyperparameters: {'max_length': 21, 'temperature': 0.6687123457289996}


## Manual Load Model

In [ ]:
# Import necessary libraries
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
import optuna

# Load pre-trained GPT-2 model and tokenizer manually
model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# Set the device to GPU if available (otherwise use CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define the objective function for Bayesian Optimization
def objective(trial):
    # Sample hyperparameters for optimization
    max_length = trial.suggest_int("max_length", 20, 100)  # Length of the output
    temperature = trial.suggest_float("temperature", 0.5, 1.5)  # Creativity level

    # Define a fixed prompt for text generation
    prompt = "Once upon a time in a world of artificial intelligence,"

    # Encode the prompt to tensor
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # Generate text using the model
    output = model.generate(
        input_ids,
        max_length=max_length,
        temperature=temperature,
        num_return_sequences=1,
        no_repeat_ngram_size=2,  # Optional: To avoid repetition
        top_k=50,  # Optional: Top-k sampling
        top_p=0.95,  # Optional: Top-p (nucleus) sampling
    )

    # Decode the generated output to text
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Toy evaluation metric: Score based on text length (adjust for your needs)
    length_score = len(generated_text)
    return -abs(length_score - 80)  # Target length around 80 characters (negated for minimization)

# Perform Bayesian Optimization with Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

# Print the best hyperparameters and score
print("Best score:", study.best_value)
print("Best hyperparameters:", study.best_params)


[I 2025-01-26 15:43:38,731] A new study created in memory with name: no-name-6fc97182-a1cc-4c8c-aa3d-5936571b731b
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9816814954989592` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention m

Best score: -12.0
Best hyperparameters: {'max_length': 20, 'temperature': 1.4721132117900049}
